In [1]:
import pandas as pd
import numpy as np

In [2]:
#%run ../gs_slimming.py
#print("="*120)
#print("gs_slimming done.")
#print("="*120)
%run flattening.py
print("="*120)
print("flattening done.")
print("="*120)


  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv
  Reports    : 54
  Zeilen     : 672
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv

  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv
  Reports    : 54
  Zeilen     : 706
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv

  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-32B-Thinking.csv
  Reports    : 54
  Zeilen     : 635
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-32B-Thinking.csv

  Flattening 54 

## Import slimmed down Gold-Standard

In [3]:
gs = pd.read_json("../gs_slim.json")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import all 3x2 + 1 = 7 Extractions

In [4]:
think1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Thinking.csv")
think2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Thinking.csv")

moe1   = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv")
moe2   = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-30B-A3B-Thinking.csv")

instr1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv")
instr2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Instruct.csv")

gepa = pd.read_csv("./PipelineB-Answers/GEPA_Qwen3-VL-32B-Thinking.csv")

df_to_merge = [
    (think1, "_t1"),
    (think2, "_t2"),
    (moe1, "_m1"),
    (moe2, "_m2"),
    (instr1, "_i1"),
    (instr2, "_i2"),
    (gepa, "_g")
]

In [6]:
# Year normalization RegEx-Script

import re

def normalize_year(raw: str, years_present: set[int] | None = None) -> tuple[int | None, str]:
    """Map a raw fiscal-year label to a calendar year. Returns (year, rule_applied)."""
    
    if isinstance(raw, float):
        if pd.isna(raw):
            return None, "unparseable"
        raw = int(raw) if raw.is_integer() else raw
    
    label = str(raw).strip().upper().removeprefix("FY").strip()

    if re.fullmatch(r"\d{4}", label):
        return int(label), "plain"
    if re.fullmatch(r"\d{2}", label):
        return 2000 + int(label), "fy_2digit"

    m = re.fullmatch(r"(\d{4})/(\d{1,4})", label)
    if m:
        left, right = m.groups()
        if len(right) == 4:
            return int(left), "range_former_year"  # 2021/2022 -> 2021
        candidates = {int(left), int(left) + 1}
        if years_present:
            hit = candidates & years_present
            if len(hit) == 1:
                return hit.pop(), "fy_end_month_context"
        return int(left), "fy_end_month_fallback"

    return None, "unparseable"

# Sammle Jahre aus den Extraktionsdaten für Kontext
years_in_extractions = set()
for df, _ in df_to_merge:
    years_in_extractions.update(df["year"].unique().tolist())

# Erstelle normalisierte Kopien der Extraktions-DataFrames
think1_ynorm = think1.copy()
think2_ynorm = think2.copy()
moe1_ynorm   = moe1.copy()
moe2_ynorm   = moe2.copy()
instr1_ynorm = instr1.copy()
instr2_ynorm = instr2.copy()
gepa_ynorm = gepa.copy()

# Aktualisiere df_to_merge mit normalisierten Versionen
df_to_merge_ynorm = [
    (think1_ynorm, "_t1"),
    (think2_ynorm, "_t2"),
    (moe1_ynorm, "_m1"),
    (moe2_ynorm, "_m2"),
    (instr1_ynorm, "_i1"),
    (instr2_ynorm, "_i2"),
    (gepa_ynorm, "_g")
]

# Normalisiere Jahre in den Kopien
for df, _ in df_to_merge_ynorm:
    df["year"], df["year_rule"] = zip(*df["year"].apply(
        normalize_year,
        years_present=years_in_extractions
    ))

## Matching Extractions to Gold-Standard Scheme

### Checking format of column "year"

In [7]:
years = set()
year_report = []

for df, _ in df_to_merge:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

['2006', '2012', '2014', '2015', '2016', '2017', '2017 pro rata', '2018', '2018/3', '2019', '2019/2020', '2019/3', '2020', '2020/2021', '2020/3', '2021', '2021/2022', '2022', 'FY 2021', 'FY 2022', 'FY16', 'FY17', 'FY18', 'FY19', 'FY20', 'FY2017', 'FY2018', 'FY2018/3', 'FY2019', 'FY2019/3', 'FY2020', 'FY2020/3', 'FY2021', 'FY2022']


In [8]:
years = set()
year_report = []

for df, _ in df_to_merge_ynorm:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

[2016, 2017, 2018, 2019, 2020, 2021, 2022, nan, 2006, 2012, 2014, 2015]


In [10]:
years_gs = sorted(gs["year"].unique().tolist())
years_gs

['2012',
 '2013',
 '2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020',
 '2021',
 '2022']

## Mapping special occurences

In [11]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



Total NaN nach Konversion: 0


In [12]:

# Prints out only if sth. goes wrong
for df, sf in df_to_merge_ynorm:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



Total NaN nach Konversion: 0


Checking if all `report_names` are the same

And checking if all `report_names` from the extractions are exactly matched in the GoldStandard

In [13]:
print("Think ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print("Moe   ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print("Instr ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print()

# print(think1["report_name"].isin(gs["report_name"]).all())
# print(gs["report_name"].isin(think1["report_name"]).all())

in_think1_not_gs  = set(think1["report_name"]) - set(gs["report_name"])
in_gs_not_think1  = set(gs["report_name"])     - set(think1["report_name"])

print(f"In think1, nicht in GS: {len(in_think1_not_gs)} ==> {"OK" if len(in_think1_not_gs) == 0 else "NOK"}")
for r in sorted(in_think1_not_gs): print(" ", r)

print(f"\nIn GS, nicht in think1: {len(in_gs_not_think1)} ==> {"OK" if len(in_gs_not_think1) == 0 else "NOK"}")
for r in sorted(in_gs_not_think1): print(" ", r)

Think ok: True
Moe   ok: True
Instr ok: True

In think1, nicht in GS: 0 ==> OK

In GS, nicht in think1: 0 ==> OK


## Merging Extraction Values and Gold-Standard

In [14]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [15]:
for df, sf in df_to_merge:
    dups = df.duplicated(subset=merge_on).sum()
    print(f"{sf}: {dups} doppelte Keys")


_t1: 147 doppelte Keys
_t2: 160 doppelte Keys
_m1: 202 doppelte Keys
_m2: 148 doppelte Keys
_i1: 190 doppelte Keys
_i2: 300 doppelte Keys
_g: 90 doppelte Keys


=> That's way I do not need a normal merge, but consolidate every extracted value, unit, and label from every dataframe (think1, think2, moe1, ...) value into a list.

Then I can validate if the gs-value is extracted or not.

These are represented as "objects" when calling merged.info() but are just python lists.

In [16]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 32 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   year             2208 non-null   str    
 2   scope            2208 non-null   str    
 3   page             490 non-null    float64
 4   value            489 non-null    float64
 5   unit             488 non-null    str    
 6   unit_normalized  488 non-null    str    
 7   label            489 non-null    str    
 8   status           2208 non-null   str    
 9   scopes_present   2208 non-null   object 
 10  years_present    2208 non-null   object 
 11  value_t1         497 non-null    object 
 12  unit_t1          497 non-null    object 
 13  label_t1         497 non-null    object 
 14  value_t2         506 non-null    object 
 15  unit_t2          506 non-null    object 
 16  label_t2         506 non-null    object 
 17  value_m1         468 non-

In [17]:
merged_ynorm = gs.copy()
merged_ynorm["year"] = merged_ynorm["year"].astype(int)

for df, sf in df_to_merge_ynorm:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged_ynorm = pd.merge(merged_ynorm, agg, on=merge_on, how="left")

merged_ynorm.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 32 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   year             2208 non-null   int64  
 2   scope            2208 non-null   str    
 3   page             490 non-null    float64
 4   value            489 non-null    float64
 5   unit             488 non-null    str    
 6   unit_normalized  488 non-null    str    
 7   label            489 non-null    str    
 8   status           2208 non-null   str    
 9   scopes_present   2208 non-null   object 
 10  years_present    2208 non-null   object 
 11  value_t1         526 non-null    object 
 12  unit_t1          526 non-null    object 
 13  label_t1         526 non-null    object 
 14  value_t2         535 non-null    object 
 15  unit_t2          535 non-null    object 
 16  label_t2         535 non-null    object 
 17  value_m1         507 non-

#### Rows where no value is present must be from Type "NaN" for better analysis; not "None" because the object is missing.

In [18]:
pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

for col in pipeline_cols:
    merged[col] = merged[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    merged_ynorm[col] = merged_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

### Rearranging Columns

In [19]:

gs_cols = [
    'report_name', 'status', 'scopes_present', 'years_present', # Per-Report granularity
    'year', 'scope',                                            # Per category
    'page', 'value', 'unit',                                    # About the value as-is in the report
    'unit_normalized', 'label',                                 # Additional information
]

pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

merged = merged[gs_cols + pipeline_cols]
merged_ynorm = merged_ynorm[gs_cols + pipeline_cols]

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 32 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   status           2208 non-null   str    
 2   scopes_present   2208 non-null   object 
 3   years_present    2208 non-null   object 
 4   year             2208 non-null   str    
 5   scope            2208 non-null   str    
 6   page             490 non-null    float64
 7   value            489 non-null    float64
 8   unit             488 non-null    str    
 9   unit_normalized  488 non-null    str    
 10  label            489 non-null    str    
 11  value_t1         497 non-null    object 
 12  value_t2         506 non-null    object 
 13  value_m1         468 non-null    object 
 14  value_m2         464 non-null    object 
 15  value_i1         490 non-null    object 
 16  value_i2         496 non-null    object 
 17  value_g          426 non-

## Saving created DataFrame

In [24]:
merged.to_json("PipelineB.json", index=False, orient="records", indent=4)
merged_ynorm.to_json("PipelineB_ynorm.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_json("PipelineB_ynorm.json", orient="records")  # Lists instant usable


## Creating Overview which scopes are present for which years of a given report (discarded)

In [22]:
# year_scope_df = (
#     gs[gs["value"].notna()]
#     .groupby(["report_name", "year"])["scope"]
#     .apply(lambda x: sorted(x.unique()))
#     .reset_index()
#     .rename(columns={"scope": "scopes"})
# )

# year_scope_df["_sort"] = year_scope_df["report_name"].str.lower() # need to circumvent ASCII case-sensitive sorting
# year_scope_df = year_scope_df.sort_values(["_sort", "year"], ignore_index=True).drop(columns="_sort")

# year_scope_df.to_csv("year_scope_df.csv", index=False)
# year_scope_df.to_json("year_scope_df.json", index=False, orient="records", indent=4)
# year_scope_df.head()
